# CUDA check
Verifies that PyTorch can see and use the GPU.

In [ ]:
import torch

print("PyTorch version :", torch.__version__)
print("CUDA built for  :", torch.version.cuda)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count       :", torch.cuda.device_count())
    print("GPU name        :", torch.cuda.get_device_name(0))
    print("cuDNN version   :", torch.backends.cudnn.version())
else:
    print("\nCUDA NOT available. Likely causes: CPU-only torch installed, "
          "missing/old NVIDIA driver, or no NVIDIA GPU.")

In [ ]:
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def bench(dev, n=4096):
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    (a @ b)  # warm-up
    if dev.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    for _ in range(5):
        c = a @ b
    if dev.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t) / 5

cpu_t = bench(torch.device("cpu"))
print(f"CPU matmul: {cpu_t*1000:.1f} ms")
if device.type == "cuda":
    gpu_t = bench(device)
    print(f"GPU matmul: {gpu_t*1000:.1f} ms  (~{cpu_t/gpu_t:.1f}x faster)")
    print("\n✅ CUDA is working.")
else:
    print("\n❌ Running on CPU only.")